
# 13 — LangChain Fundamentals and LCEL

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

## Course Material Covered

This notebook consolidates the following lessons:

1. `langchain_installation.ipynb`
2. `chat_models.ipynb`
3. `chat_models_prompt_templates.ipynb`
4. `chat_models_output_parsing.ipynb`
5. `chat_models_streaming.ipynb`
6. `LCEL_runnable_interface.ipynb`
7. `LCEL_multiple_chains.ipynb`
8. `LCEL_conditional_logic_branching_and_merging.ipynb`
9. `LCEL_configurable_fields.ipynb`
10. `LCEL_chat_models_itemgetter_rag.ipynb`
11. `chat_models_document_loaders_and_text_splitting.ipynb`
12. `chat_models_tagging_documents.ipynb`
13. `chat_models_summarization_chains.ipynb`
14. `vectorstores_and_the_indexing_api.ipynb`

Conversation memory and agents are covered in Notebook 14.

## Goal

Understand how LangChain organizes reusable components
for building language-model applications.

We will learn:

- Chat messages and reusable prompt templates.
- Structured output and validation.
- The Runnable interface and LCEL.
- Sequential, parallel and conditional workflows.
- Runtime configuration.
- Document loading and text splitting.
- Retrieval and document-processing concepts.
- Summarization strategies.
- The difference between vector search and index maintenance.

The main executable demonstrations use local Python
functions and LangChain components.

This isolates workflow behavior from model behavior
and makes the examples independently testable.



## 1. What Are LangChain and LCEL?

**LangChain** is a framework for building applications
that combine language models with prompts, tools,
retrievers and other components.

**LCEL** stands for LangChain Expression Language.

LCEL connects operations using the pipe operator (`|`).

For example:

Input → Format prompt → Call model → Parse output

A **Runnable** is a component that accepts input
and produces output.

Common Runnable methods:

- `invoke(input)` — process one input.
- `batch(inputs)` — process multiple inputs.
- `stream(input)` — produce output incrementally,
  when supported by the components.
- `with_config(...)` — supply execution settings.

A Runnable can be a model, prompt, parser or ordinary
Python function wrapped with `RunnableLambda`.

LangChain provides the building blocks. LCEL defines
how they are composed.


In [1]:

%pip install -q "langchain-core>=1,<2" "langchain-text-splitters>=1,<2" "pydantic>=2,<3"


In [2]:

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)
from langchain_core.prompts import ChatPromptTemplate

messages = [
    SystemMessage(content="You explain programming simply."),
    HumanMessage(content="What is a Python dictionary?")
]

print("Conversation messages:")
for message in messages:
    print(type(message).__name__, ":", message.content)


prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain {subject} to beginners."),
    ("human", "Explain {concept} using a short example.")
])

formatted = prompt.invoke({
    "subject": "Python",
    "concept": "dictionaries"
})

print("\nFormatted prompt:")
for message in formatted.messages:
    print(type(message).__name__, ":", message.content)

print("\nModel requests: 0")


Conversation messages:
SystemMessage : You explain programming simply.
HumanMessage : What is a Python dictionary?

Formatted prompt:
SystemMessage : You explain Python to beginners.
HumanMessage : Explain dictionaries using a short example.

Model requests: 0



## 2. Sequential Workflows with RunnableLambda

A sequential workflow passes the output of one
operation into the next.

Example:

Input 5 → Add 1 → Multiply by 2 → Output 12

`RunnableLambda` wraps an ordinary Python function
so it supports LangChain's Runnable interface.

The `|` operator creates a sequence.

This is the same core idea used in multi-step
LLM pipelines, except our example runs locally.

A real LLM chain might perform:

Question → Extract location → Identify country

Each model-backed step may require a separate request.


In [3]:

from langchain_core.runnables import RunnableLambda

add_one = RunnableLambda(lambda x: x + 1)
multiply_two = RunnableLambda(lambda x: x * 2)

chain = add_one | multiply_two

result = chain.invoke(5)
assert result == 12

print("Single input:", result)

batch_results = chain.batch([1, 2, 3])
assert batch_results == [4, 6, 8]

print("Batch inputs:", batch_results)

print("\nSequential execution:")
print("5 -> 6 -> 12")

# Streaming depends on whether components support
# incremental output. This simple chain emits one result.
print("\nRunnable stream:")
for chunk in chain.stream(5):
    print("Chunk:", chunk)


Single input: 12
Batch inputs: [4, 6, 8]

Sequential execution:
5 -> 6 -> 12

Runnable stream:
Chunk: 12



## 3. Parallel Execution, Branching and Merging

Three useful LCEL building blocks are:

**RunnableParallel**

Runs independent operations using the same input
and returns their results in a dictionary.

**RunnableBranch**

Selects an operation according to a condition.

**Merging**

Combines the results of separate processing steps
into one final output.

These techniques correspond to workflows we have
previously implemented directly in Python.

Here, LangChain provides a consistent interface
for composing the steps.


In [4]:

from langchain_core.runnables import (
    RunnableParallel,
    RunnableBranch
)

parallel = RunnableParallel(
    uppercase=RunnableLambda(lambda text: text.upper()),
    length=RunnableLambda(lambda text: len(text)),
    word_count=RunnableLambda(
        lambda text: len(text.split())
    )
)

parallel_result = parallel.invoke(
    "Your order is delayed"
)

print("Parallel results:", parallel_result)

assert parallel_result["length"] == 21
assert parallel_result["word_count"] == 4


merge = RunnableLambda(
    lambda data: (
        f"{data['uppercase']} "
        f"({data['word_count']} words)"
    )
)

combined_chain = parallel | merge
print("Merged result:", combined_chain.invoke(
    "Your order is delayed"
))


router = RunnableBranch(
    (
        lambda message: "damaged" in message.lower(),
        RunnableLambda(lambda message: "damage_team")
    ),
    (
        lambda message: "late" in message.lower(),
        RunnableLambda(lambda message: "delivery_team")
    ),
    RunnableLambda(lambda message: "general_team")
)

assert router.invoke("My item arrived damaged") == "damage_team"
assert router.invoke("My delivery is late") == "delivery_team"
assert router.invoke("Hello") == "general_team"

print("\nBranching tests passed.")


Parallel results: {'uppercase': 'YOUR ORDER IS DELAYED', 'length': 21, 'word_count': 4}
Merged result: YOUR ORDER IS DELAYED (4 words)

Branching tests passed.


In [5]:

from operator import itemgetter

from langchain_core.runnables import (
    RunnablePassthrough,
    ConfigurableField
)
from langchain_core.prompts import PromptTemplate

customer = {
    "name": "Maya",
    "order_id": "1042",
    "issue": "late delivery"
}

get_issue = itemgetter("issue")
print("Selected field:", get_issue(customer))


enrich = RunnablePassthrough.assign(
    normalized_issue=lambda data: data["issue"].upper()
)

enriched = enrich.invoke(customer)

assert enriched["name"] == "Maya"
assert enriched["normalized_issue"] == "LATE DELIVERY"

print("Enriched dictionary:", enriched)


base_prompt = PromptTemplate.from_template(
    "Explain {topic} simply."
)

alternate_prompt = PromptTemplate.from_template(
    "Explain {topic} using a technical example."
)

configurable_prompt = base_prompt.configurable_alternatives(
    ConfigurableField(id="prompt_style"),
    default_key="simple",
    technical=alternate_prompt
)

simple = configurable_prompt.invoke({"topic": "RAG"})

technical = configurable_prompt.with_config(
    configurable={"prompt_style": "technical"}
).invoke({"topic": "RAG"})

print("\nDefault prompt:", simple.text)
print("Alternate prompt:", technical.text)


Selected field: late delivery
Enriched dictionary: {'name': 'Maya', 'order_id': '1042', 'issue': 'late delivery', 'normalized_issue': 'LATE DELIVERY'}

Default prompt: Explain RAG simply.
Alternate prompt: Explain RAG using a technical example.



## 4. Documents, Loading and Text Splitting

LangChain's `Document` class separates:

- `page_content`: The document text.
- `metadata`: Information such as source or category.

A document loader converts files, websites or other
sources into `Document` objects.

A text splitter divides longer documents into chunks.

Why split?

- A document may be too long for a model prompt.
- Retrieval often works better with focused passages.
- Individual chunks can be summarized or indexed.

A common splitter is `RecursiveCharacterTextSplitter`.

It attempts to split at natural boundaries such
as paragraphs and spaces before resorting to
smaller units.

`chunk_overlap` allows neighboring chunks to
share some content, preserving context across
their boundaries.


In [6]:

from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

sample_text = (
    "Returns Policy\n"
    "Customers may return unused products within 30 days. "
    "A receipt is required.\n\n"
    "Delivery Policy\n"
    "Standard shipping usually takes three to five "
    "business days. Customers can report late deliveries.\n\n"
    "Damaged Items Policy\n"
    "Customers should contact support if an item "
    "arrives damaged. Support may arrange a replacement."
)

source_path = Path("sample_support_policies.txt")
source_path.write_text(sample_text, encoding="utf-8")

loaded_document = Document(
    page_content=source_path.read_text(encoding="utf-8"),
    metadata={"source": source_path.name}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=110,
    chunk_overlap=15
)

chunks = splitter.split_documents([loaded_document])

print("Source documents:", 1)
print("Generated chunks:", len(chunks))

for number, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {number}:")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)

assert len(chunks) > 1


Source documents: 1
Generated chunks: 5

Chunk 1:
Returns Policy
Customers may return unused products within 30 days. A receipt is required.
Metadata: {'source': 'sample_support_policies.txt'}

Chunk 2:
Delivery Policy
Metadata: {'source': 'sample_support_policies.txt'}

Chunk 3:
Standard shipping usually takes three to five business days. Customers can report late deliveries.
Metadata: {'source': 'sample_support_policies.txt'}

Chunk 4:
Damaged Items Policy
Metadata: {'source': 'sample_support_policies.txt'}

Chunk 5:
Customers should contact support if an item arrives damaged. Support may arrange a replacement.
Metadata: {'source': 'sample_support_policies.txt'}



## 5. Document Tagging, Retrieval and RAG

Document tagging adds descriptive metadata.

Examples:

- Topic: returns, delivery, damaged items.
- Language: English.
- Sentiment: positive, neutral, negative.

In the course, an LLM assigns tags to documents
retrieved from a website sitemap.

Here we use deterministic labels so the tagging
logic can be inspected directly.

Retrieval is different from tagging.

A retriever searches available documents and returns
relevant passages for a question.

RAG stands for Retrieval-Augmented Generation:

Question → Retrieve evidence → Build prompt → Model answer

This notebook demonstrates retrieval and prompt
construction locally. It does not claim that keyword
matching is equivalent to semantic vector search.


In [7]:

from langchain_core.runnables import RunnableLambda

policies = [
    Document(
        page_content="Unused products may be returned within 30 days.",
        metadata={"source": "returns.txt", "topic": "returns"}
    ),
    Document(
        page_content="Standard shipping usually takes 3–5 business days.",
        metadata={"source": "delivery.txt", "topic": "delivery"}
    ),
    Document(
        page_content="Contact support to request a replacement for damaged items.",
        metadata={"source": "damage.txt", "topic": "damage"}
    )
]


def retrieve_policy(question):
    text = question.lower()

    if "return" in text:
        selected_topic = "returns"
    elif "damage" in text or "broken" in text:
        selected_topic = "damage"
    else:
        selected_topic = "delivery"

    return [
        doc for doc in policies
        if doc.metadata["topic"] == selected_topic
    ]


retriever = RunnableLambda(retrieve_policy)

rag_inputs = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough()
)

def format_context(data):
    passages = "\n".join(
        doc.page_content for doc in data["context"]
    )

    return {
        "question": data["question"],
        "context": passages
    }


rag_prompt = ChatPromptTemplate.from_template(
    "Use only the following policy context.\n"
    "Context: {context}\n"
    "Question: {question}\n"
    "If the context is insufficient, say so."
)

rag_chain = (
    rag_inputs
    | RunnableLambda(format_context)
    | rag_prompt
)

formatted_rag = rag_chain.invoke(
    "Can I return an unused item?"
)

print(formatted_rag.to_string())
print("\nThis constructs a retrieval-grounded prompt.")
print("It does not generate an AI answer.")


Human: Use only the following policy context.
Context: Unused products may be returned within 30 days.
Question: Can I return an unused item?
If the context is insufficient, say so.

This constructs a retrieval-grounded prompt.
It does not generate an AI answer.



## 6. Summarization Strategies

The course introduces three approaches.

### Stuff

Combine all document text into one prompt and
request a summary.

Advantages:
- Simple.
- Often sufficient for small documents.

Limitation:
- The combined text must fit the model's context.

### Map-Reduce

First summarize each chunk independently (map).

Then combine the partial summaries into one
summary (reduce).

Advantages:
- Supports independent chunk processing.
- Useful for large document collections.

Limitations:
- Usually requires multiple model calls.
- Important cross-chunk context may be lost.

### Refine

Summarize the first chunk, then revise the
summary using each subsequent chunk.

Advantages:
- Builds a summary incrementally.

Limitations:
- Sequential processing.
- Earlier information may be lost or distorted.

The uploaded course file demonstrates Stuff and
Map-Reduce execution, but does not show a complete
successful Refine execution.

The next cell demonstrates the three data-flow
patterns locally, not their semantic summarization quality.


In [8]:

sections = [
    "Returns: unused products are accepted within 30 days.",
    "Delivery: standard shipping takes 3–5 business days.",
    "Damage: contact support for replacement options."
]

# Stuff: one combined input
stuff_input = "\n".join(sections)

# Map: each chunk processed independently
def map_section(text):
    return text.strip()

mapped = [map_section(section) for section in sections]

# Reduce: combine mapped outputs
reduced = " | ".join(mapped)

# Refine: accumulate information sequentially
refined = ""

for section in sections:
    refined = (
        section if not refined
        else refined + " | " + section
    )

print("STUFF INPUT:\n", stuff_input)
print("\nMAP-REDUCE FLOW:\n", reduced)
print("\nREFINE FLOW:\n", refined)

assert len(mapped) == 3

print("\nThese are workflow demonstrations,")
print("not AI-generated summaries.")


STUFF INPUT:
 Returns: unused products are accepted within 30 days.
Delivery: standard shipping takes 3–5 business days.
Damage: contact support for replacement options.

MAP-REDUCE FLOW:
 Returns: unused products are accepted within 30 days. | Delivery: standard shipping takes 3–5 business days. | Damage: contact support for replacement options.

REFINE FLOW:
 Returns: unused products are accepted within 30 days. | Delivery: standard shipping takes 3–5 business days. | Damage: contact support for replacement options.

These are workflow demonstrations,
not AI-generated summaries.



## 7. Vector Stores and Index Maintenance

A vector store holds numerical representations
called embeddings.

Embeddings allow a search system to compare the
meaning of documents and queries using similarity
calculations.

The course demonstrates similarity search with FAISS
and index maintenance with Chroma and a record manager.

These are separate responsibilities:

**Vector search**

Find documents that are similar to a query.

**Index maintenance**

Keep stored documents synchronized with their sources.

For example, if a document changes from
"three years of experience" to
"seven years of experience", the old indexed
content should not remain the only searchable version.

A record manager helps identify documents that are
new, unchanged, modified or no longer present.

The indexing API's cleanup modes include:

- `none`: no stale-document cleanup.
- `incremental`: clean up outdated records as
  changed source documents are processed.
- `full`: reconcile the complete indexed set,
  removing records absent from a full indexing run.

The choice depends on whether the current run
contains a complete source inventory.

Below we demonstrate change detection locally.
This is an illustration of the indexing concept,
not a replacement for LangChain's indexing API.


In [9]:

import hashlib


def content_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


index_registry = {}


def update_registry(documents):
    changes = {
        "added": [],
        "updated": [],
        "unchanged": []
    }

    for source_id, content in documents.items():
        new_hash = content_hash(content)

        if source_id not in index_registry:
            changes["added"].append(source_id)

        elif index_registry[source_id] != new_hash:
            changes["updated"].append(source_id)

        else:
            changes["unchanged"].append(source_id)

        index_registry[source_id] = new_hash

    return changes


original = {
    "profile": "James has three years of experience.",
    "policy": "Returns accepted within 30 days."
}

first = update_registry(original)
second = update_registry(original)

modified = {
    **original,
    "profile": "James has seven years of experience."
}

third = update_registry(modified)

print("First indexing:", first)
print("Second indexing:", second)
print("After update:", third)

assert set(first["added"]) == {"profile", "policy"}
assert len(second["unchanged"]) == 2
assert third["updated"] == ["profile"]

print("\nIndex change-detection tests passed.")


First indexing: {'added': ['profile', 'policy'], 'updated': [], 'unchanged': []}
Second indexing: {'added': [], 'updated': [], 'unchanged': ['profile', 'policy']}
After update: {'added': [], 'updated': ['profile'], 'unchanged': ['policy']}

Index change-detection tests passed.


In [10]:

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class SupportTag(BaseModel):
    category: str = Field(
        description="Support request category"
    )
    priority: str = Field(
        description="Priority label"
    )


parser = PydanticOutputParser(
    pydantic_object=SupportTag
)

example_response = """
{
    "category": "delivery",
    "priority": "normal"
}
"""

parsed = parser.parse(example_response)

assert isinstance(parsed, SupportTag)
assert parsed.category == "delivery"
assert parsed.priority == "normal"

print("Parsed category:", parsed.category)
print("Parsed priority:", parsed.priority)
print("Output parsing test passed.")


Parsed category: delivery
Parsed priority: normal
Output parsing test passed.



## 8. Key Learnings and Limitations

### LangChain Fundamentals

- Chat messages represent system, user and AI roles.
- Prompt templates make instructions reusable.
- Output parsing converts model responses into
  structured application data.
- Streaming returns pieces of an output when
  supported by the underlying components.

### LCEL

- `RunnableLambda` wraps Python functions.
- `invoke()` runs one input.
- `batch()` processes multiple inputs.
- `RunnableSequence` connects operations.
- `RunnableParallel` processes independent branches.
- `RunnableBranch` supports conditional execution.
- `RunnablePassthrough` preserves or enriches inputs.
- Configurable fields select runtime alternatives.

### Document Processing

- Document loaders create standard Document objects.
- Text splitters divide long material into chunks.
- Tagging creates metadata.
- Retrieval selects relevant context.
- RAG combines retrieval with model generation.
- Stuff, Map-Reduce and Refine describe different
  summarization data flows.
- Vector stores support similarity search.
- Index managers track changes and cleanup.

### Limitations

- The executable workflows are mostly deterministic
  Python and LCEL demonstrations.
- Keyword retrieval is not semantic vector search.
- The summarization examples demonstrate control
  flow rather than LLM summarization quality.
- The hash registry illustrates change detection,
  not the full LangChain indexing API.
- No provider-specific model behavior is benchmarked.

### Conclusion

This notebook explains how LangChain components can
be assembled into reusable workflows.

The main learning outcome is understanding how data
moves between operations, how branches are selected,
and how document context is prepared for model use.

The next notebook builds on these foundations using
LangGraph's explicit state and workflow transitions.
